# MiniProject2 - Part 2-4: Visualization, Gap Analysis, Interpretation

NetID: tlopez7

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df_all = pd.read_csv('tlopez7_project_summary.csv', sep=';')
df_all['date'] = pd.to_datetime(df_all['time'], unit='s')
df_all['month'] = df_all['date'].dt.to_period('M')

projects = df_all['project_wocid'].unique()

for proj in projects:
    dfp = df_all[df_all['project_wocid'] == proj]
    monthly = dfp.groupby('month').size()

    # fill in zero-commit months so there are no gaps in the timeline
    full_range = pd.period_range(monthly.index.min(), monthly.index.max(), freq='M')
    monthly = monthly.reindex(full_range, fill_value=0)

    ts_df = pd.DataFrame({'Month': monthly.index.astype(str), '#Commits': monthly.values})
    ts_df.to_csv(f'tlopez7_commits_timeseries_{proj}.csv', sep=';', index=False)

    plt.figure(figsize=(10, 5))
    plt.plot(ts_df['Month'], ts_df['#Commits'], marker='o', linestyle='-')
    plt.xlabel('Time (YYYY-MM)')
    plt.ylabel('Number of Commits')
    plt.title(f'Commit Activity: {proj}')
    plt.grid(True)

    # thin out x-axis labels so long histories aren't unreadable
    n = max(1, len(ts_df) // 15)
    plt.xticks(range(0, len(ts_df), n), ts_df['Month'][::n], rotation=45)

    plt.tight_layout()
    plt.savefig(f'tlopez7_timeseries_{proj}.png', dpi=300)
    plt.close()

    print(f"{proj}: {len(ts_df)} months -> CSV + PNG saved")

gavinsimpson_permute: 194 months -> CSV + PNG saved
openclimatefix_satip: 60 months -> CSV + PNG saved
juliaopt_convex.jl: 138 months -> CSV + PNG saved
eljost_pysisyphus: 100 months -> CSV + PNG saved
simonvh_genomepy: 106 months -> CSV + PNG saved
brentp_cyvcf2: 123 months -> CSV + PNG saved
dicarlolab-delft_pycqed_py3: 117 months -> CSV + PNG saved
benchmarking-initiative_benchmark-models-petab: 70 months -> CSV + PNG saved
gsdlab_clafer: 170 months -> CSV + PNG saved
acellera_htmd: 118 months -> CSV + PNG saved


In [2]:
def gap_stats(ts_csv, min_gap=3):
    ts = pd.read_csv(ts_csv, sep=';')
    zero_mask = (ts['#Commits'] == 0).tolist()

    gaps = []
    cur_start, cur_len = None, 0
    for i, is_zero in enumerate(zero_mask):
        if is_zero:
            if cur_len == 0:
                cur_start = i
            cur_len += 1
        else:
            if cur_len > 0:
                gaps.append((cur_start, i - 1, cur_len))
            cur_len = 0
    if cur_len > 0:
        gaps.append((cur_start, len(zero_mask) - 1, cur_len))

    longest = max(gaps, key=lambda g: g[2]) if gaps else (None, None, 0)
    gap_start = ts['Month'].iloc[longest[0]] if longest[0] is not None else None
    gap_end = ts['Month'].iloc[longest[1]] if longest[1] is not None else None
    commits_after = ts['#Commits'].iloc[longest[1]+1:].sum() if longest[1] is not None else ts['#Commits'].sum()
    num_gaps = sum(1 for g in gaps if g[2] >= min_gap)

    return {
        'LongestGapStart': gap_start, 'LongestGapEnd': gap_end,
        'LongestGapLength': longest[2], 'NumCommitsAfterLastGap': commits_after,
        'NumberOfGapsInTimeline': num_gaps,
    }

print(gap_stats('tlopez7_commits_timeseries_gavinsimpson_permute.csv'))

{'LongestGapStart': '2019-04', 'LongestGapEnd': '2021-10', 'LongestGapLength': 31, 'NumCommitsAfterLastGap': np.int64(64), 'NumberOfGapsInTimeline': 14}


## Interpretation: gavinsimpson_permute

**ActivityPattern: Declining**

The project shows highly irregular, bursty activity from 2010-2016, 
including two major spikes (~104 commits in mid-2013, ~64 commits in 
late 2015), surrounded by many quiet months even during its "active" 
years — 14 separate gaps of 3+ months occur across the whole timeline. 
After 2016, activity drops off almost entirely, with only small, 
infrequent bursts remaining (2019, 2021-2022, 2024, 2025).

The single longest gap ran from 2019-04 to 2021-10 (31 months). Only 
64 of the project's 552 total commits (~12%) occurred after this gap 
ended, confirming most of the project's real activity was already 
behind it — consistent with a declining trajectory rather than a 
genuine revival.

In [3]:
results = []
for proj in projects:
    stats = gap_stats(f'tlopez7_commits_timeseries_{proj}.csv')
    stats['project'] = proj
    results.append(stats)

df_gaps = pd.DataFrame(results)
print(df_gaps)

  LongestGapStart LongestGapEnd  LongestGapLength  NumCommitsAfterLastGap  \
0         2019-04       2021-10                31                      64   
1         2021-07       2021-08                 2                    1813   
2         2023-03       2023-06                 4                     973   
3         2017-10       2017-10                 1                    6986   
4         2023-12       2024-08                 9                      49   
5         2024-10       2025-08                11                      18   
6         2024-12       2025-01                 2                      27   
7         2020-07       2020-09                 3                     585   
8         2019-06       2023-07                50                      20   
9         2023-07       2023-09                 3                     181   

   NumberOfGapsInTimeline                                         project  
0                      14                            gavinsimpson_permute  


## Interpretation: openclimatefix_satip

**ActivityPattern: Irregular**

Activity is bursty and unpredictable from the start — big spikes (up to 
170 commits/month) mixed with near-dead months, with no clear rising or 
falling trend across the full ~5 years. Notably, there are zero gaps of 
3+ months anywhere in its history — even the quietest stretches got at 
least an occasional commit. Most of the project's activity is concentrated 
after 2021 (86% of all commits came after its longest gap, a short 
2-month dip in mid-2021), but the volatility never settles into a 
consistent rhythm, and the last few months (2025) show a clear taper.

## Interpretation: juliaopt_convex.jl

**ActivityPattern: Irregular**

Years of modest, choppy activity (mostly under 60 commits/month) suddenly 
give way to a burst of extreme spikes (175-360 commits/month) in 2023, 
right after the project's longest gap (2023-03 to 2023-06, 4 months) — 
then the project goes almost silent again by 2024. This looks like a 
short-lived revival (possibly a major version push or refactor) rather 
than a genuine sustained comeback, since activity collapses again shortly 
after.

## Interpretation: eljost_pysisyphus

**ActivityPattern: Irregular**

Consistently high and volatile activity for nearly its entire ~8-year 
history, swinging between roughly 50-300 commits per month with no 
sustained calm periods (0 gaps of 3+ months, and 97% of all commits came 
after its one brief 1-month dip in 2017). The only notable change is a 
visible slowdown in the final months of 2025, where commit counts drop 
into the single digits for the first time.

## Interpretation: simonvh_genomepy

**ActivityPattern: Declining**

Strong, spiky activity from 2019-2022 (including a 127-commit peak) gives 
way to a long quiet stretch, then effectively stops after 2023 — only 49 
of 1601 total commits (about 3%) occurred after its longest gap 
(2023-12 to 2024-08, 9 months). Five separate 3+ month gaps occur across 
the timeline, clustering more frequently toward the end.

## Interpretation: brentp_cyvcf2

**ActivityPattern: Irregular**

Sparse and inconsistent for essentially its whole life, not just recently 
— even its busiest years (2015-2023) show frequent zero-commit months 
punctuated by occasional spikes. Only 2.4% of commits happened after its 
longest gap (2024-10 to 2025-08, 11 months), and 3 separate 3+ month gaps 
occur throughout, suggesting a project maintained in occasional bursts by 
a small group rather than continuous development.

## Interpretation: dicarlolab-delft_pycqed_py3

**ActivityPattern: Declining**

A textbook decline: a massive peak in 2017 (over 400 commits in one 
month) gradually tapers to a low, thin trickle by 2023-2025. Interestingly, 
this project has zero formal 3+ month gaps in its entire history — even 
in its quietest years, at least one commit sneaks in every month or two, 
which is why the "longest gap" algorithm only found a tiny 2-month dip 
right at the very end (2024-12 to 2025-01), after which just 27 of 8771 
total commits (0.3%) occurred. The gap-detection numbers undersell how 
dead this project has become; the chart tells the real story.

## Interpretation: benchmarking-initiative_benchmark-models-petab

**ActivityPattern: Cyclical**

Recurring bursts of 20-70 commits every few months, with no clear overall 
rising or declining trend — the project is just as active at the end of 
the timeline (a 32-commit month in late 2025) as it was in 2021. Its 
longest gap (2020-07 to 2020-09, 3 months) happened early, before 81% of 
all commits occurred, suggesting the project only found its steady 
rhythm after a brief startup lull.

## Interpretation: gsdlab_clafer

**ActivityPattern: Declining**

The most extreme decline of the 10 — heavy activity (two peaks over 100 
and 300 commits) concentrated entirely in 2011-2015, followed by a 
50-month gap (2019-06 to 2023-07), the longest of any project studied. 
Only 20 of 2089 total commits (about 1%) occurred after that gap ended, 
and small trickles since 2023 look like maintenance rather than active 
development.

## Interpretation: acellera_htmd

**ActivityPattern: Declining**

A clear boom-and-fade shape: intense activity (150-375 commits/month) 
from 2016-2019, then a steady falloff into a persistent low trickle 
(mostly under 20 commits/month) for 2020-2025. Only 3.2% of commits 
happened after its longest gap (2024-10 to 2025-08, 11 months), 
consistent with a project past its main development phase but still 
lightly maintained.

In [4]:
def get_gap_commits(project, gap_start, gap_end, n=10):
    dfp = df_all[df_all['project_wocid'] == project].sort_values('time').copy()
    dfp['month'] = pd.to_datetime(dfp['time'], unit='s').dt.to_period('M')

    gap_start_period = pd.Period(gap_start, freq='M')
    gap_end_period = pd.Period(gap_end, freq='M')

    before = dfp[dfp['month'] < gap_start_period].tail(n).copy()
    after = dfp[dfp['month'] > gap_end_period].head(n).copy()

    before['pre_post'] = 'pre'
    after['pre_post'] = 'post'

    return pd.concat([before, after])

all_gap_commits = []
for _, row in df_gaps.iterrows():
    proj = row['project']
    gc = get_gap_commits(proj, row['LongestGapStart'], row['LongestGapEnd'])
    all_gap_commits.append(gc)

df_gap_commits = pd.concat(all_gap_commits)
df_gap_commits = df_gap_commits[['pre_post', 'project_wocid', 'commit_sha1', 'author', 'time', 'commit message']]
df_gap_commits.to_csv('tlopez7_project_gap_commits.csv', sep=';', index=False)

print(df_gap_commits.groupby(['project_wocid', 'pre_post']).size())

project_wocid                                   pre_post
acellera_htmd                                   post        10
                                                pre         10
benchmarking-initiative_benchmark-models-petab  post        10
                                                pre         10
brentp_cyvcf2                                   post        10
                                                pre         10
dicarlolab-delft_pycqed_py3                     post        10
                                                pre         10
eljost_pysisyphus                               post        10
                                                pre         10
gavinsimpson_permute                            post        10
                                                pre         10
gsdlab_clafer                                   post        10
                                                pre         10
juliaopt_convex.jl                              post        1

In [5]:
for proj in projects:
    dfp = df_gap_commits[df_gap_commits['project_wocid'] == proj]
    print(f"\n=== {proj} ===")
    print("-- PRE --")
    for msg in dfp[dfp['pre_post'] == 'pre']['commit message']:
        print(" ", msg.strip()[:100])
    print("-- POST --")
    for msg in dfp[dfp['pre_post'] == 'post']['commit message']:
        print(" ", msg.strip()[:100])


=== gavinsimpson_permute ===
-- PRE --
  fix tests of shuffle affected by change to RNG in r 3.6.0
  tweak a travis setup as need to be checking this on R-devel
  set RNG from 3.5 in tests
  bump version
  set RNGversion to 3.5
  set RNGversion to 3.5
  rebuild reference example output
  set RNGversion to 3.5
  rebuild reference example output
  Merge branch 'issue-25':

These commits fix issues related to the use of a new, default, RNG method

-- POST --
  ignore revdeps
  fix for #28
  docmument problem and fix for #28 and bump version
  build the R tarball with --compact-vignettes
  add GH action to check package
  add a latex stack
  convert the vignette to rmarkdown/html
  ignore the GH folder
  rebuild reference output
  don't need latex any longer

=== openclimatefix_satip ===
-- PRE --
  added offline summary and validation
  cleaned up time-series extraction nb
  updated settings.ini and mkdocs.yml to reflect repo move
  cleaned up zarr nb docs
  cleaned up reprojection nb
  

In [6]:
before_after_themes = {
    'gavinsimpson_permute': ('Bug fixes:Dependency updates', 'Bug fixes:Documentation updates'),
    'openclimatefix_satip': ('Documentation updates:Feature development', 'Bug fixes:Other'),
    'juliaopt_convex.jl': ('Feature development:Release', 'Other:Bug fixes'),
    'eljost_pysisyphus': ('Feature development:Other', 'Feature development:Bug fixes'),
    'simonvh_genomepy': ('Bug fixes:Release', 'Bug fixes:Dependency updates'),
    'brentp_cyvcf2': ('Dependency updates:Feature development', 'Dependency updates:Bug fixes'),
    'dicarlolab-delft_pycqed_py3': ('Feature development:Bug fixes', 'Dependency updates:Feature development'),
    'benchmarking-initiative_benchmark-models-petab': ('Feature development:Bug fixes', 'Feature development:Other'),
    'gsdlab_clafer': ('Dependency updates:Release', 'Dependency updates:Other'),
    'acellera_htmd': ('Bug fixes:Feature development', 'Bug fixes:Dependency updates'),
}

activity_patterns = {
    'gavinsimpson_permute': 'Declining', 'openclimatefix_satip': 'Irregular',
    'juliaopt_convex.jl': 'Irregular', 'eljost_pysisyphus': 'Irregular',
    'simonvh_genomepy': 'Declining', 'brentp_cyvcf2': 'Irregular',
    'dicarlolab-delft_pycqed_py3': 'Declining', 'benchmarking-initiative_benchmark-models-petab': 'Cyclical',
    'gsdlab_clafer': 'Declining', 'acellera_htmd': 'Declining',
}

df_gaps['ActivityPattern'] = df_gaps['project'].map(activity_patterns)
df_gaps['BeforeThemes'] = df_gaps['project'].map(lambda p: before_after_themes[p][0])
df_gaps['AfterThemes'] = df_gaps['project'].map(lambda p: before_after_themes[p][1])

df_gaps.to_csv('tlopez7_project_stats.csv', index=False)
print(df_gaps)

  LongestGapStart LongestGapEnd  LongestGapLength  NumCommitsAfterLastGap  \
0         2019-04       2021-10                31                      64   
1         2021-07       2021-08                 2                    1813   
2         2023-03       2023-06                 4                     973   
3         2017-10       2017-10                 1                    6986   
4         2023-12       2024-08                 9                      49   
5         2024-10       2025-08                11                      18   
6         2024-12       2025-01                 2                      27   
7         2020-07       2020-09                 3                     585   
8         2019-06       2023-07                50                      20   
9         2023-07       2023-09                 3                     181   

   NumberOfGapsInTimeline                                         project  \
0                      14                            gavinsimpson_permute  

In [7]:
github_data = {
    'gavinsimpson_permute': (24, 5, '2026-09-05'),
    'openclimatefix_satip': (44, 34, '2025-08-27'),
    'juliaopt_convex.jl': (597, 123, '2026-08-19'),
    'eljost_pysisyphus': (134, 49, '2025-10-29'),
    'simonvh_genomepy': (411, 41, '2026-06-09'),
    'brentp_cyvcf2': (449, 80, '2026-09-10'),
    'dicarlolab-delft_pycqed_py3': (74, 46, '2025-07-31'),
    'benchmarking-initiative_benchmark-models-petab': (35, 20, '2026-09-19'),
    'gsdlab_clafer': (49, 15, '2026-07-15'),
    'acellera_htmd': (276, 60, '2026-09-23'),
}

woc_agg = df_all.groupby('project_wocid').agg(
    ncommits=('commit_sha1', 'count'),
    nauthors=('author', 'nunique'),
    max_time=('time', 'max'),
    min_time=('time', 'min'),
).reset_index()
woc_agg['from'] = pd.to_datetime(woc_agg['min_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_agg['to'] = pd.to_datetime(woc_agg['max_time'], unit='s').dt.strftime('%Y-%m-%d')

df_stats = df_gaps.rename(columns={'project': 'Project'}).merge(
    woc_agg[['project_wocid', 'ncommits', 'nauthors', 'from', 'to']],
    left_on='Project', right_on='project_wocid'
).drop(columns='project_wocid')

df_stats['nstars'] = df_stats['Project'].map(lambda p: github_data[p][0])
df_stats['nforks'] = df_stats['Project'].map(lambda p: github_data[p][1])
df_stats['lastGHCommitDate'] = df_stats['Project'].map(lambda p: github_data[p][2])

# reorder to match the README's column order
df_stats = df_stats[[
    'Project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate',
    'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap',
    'ActivityPattern', 'NumberOfGapsInTimeline', 'BeforeThemes', 'AfterThemes'
]]

df_stats.to_csv('tlopez7_project_stats.csv', index=False)
print(df_stats)

                                          Project  ncommits  nauthors  \
0                            gavinsimpson_permute       552         6   
1                            openclimatefix_satip      2104        49   
2                              juliaopt_convex.jl      2851       113   
3                               eljost_pysisyphus      7172        38   
4                                simonvh_genomepy      1601        18   
5                                   brentp_cyvcf2       753        61   
6                     dicarlolab-delft_pycqed_py3      8771       190   
7  benchmarking-initiative_benchmark-models-petab       722        56   
8                                   gsdlab_clafer      2089        29   
9                                   acellera_htmd      5610        51   

         from          to  nstars  nforks lastGHCommitDate LongestGapStart  \
0  2010-01-04  2026-02-05      24       5       2026-09-05         2019-04   
1  2020-11-02  2025-10-06      44      3

In [8]:
recent_commits = {}
for proj in projects:
    dfp = df_all[df_all['project_wocid'] == proj].sort_values('time')
    recent_commits[proj] = dfp.tail(10)['commit message'].tolist()
    print(f"=== {proj} ===")
    for m in recent_commits[proj]:
        print(" ", m.strip()[:100])
    print()

=== gavinsimpson_permute ===
  Deploying to gh-pages from @ gavinsimpson/permute@bf6df8f3952c161c55090b9ae9e55373a83c0beb 🚀
  ignore a file when building
  latest cran comments
  Deploying to gh-pages from @ gavinsimpson/permute@ccf61b38c09fed2c21a11292a71fe7518b252e4c 🚀
  try harder to avoid floating point issues like #41
  add some tests to confirm #41 is solved
  bump patch version
  Deploying to gh-pages from @ gavinsimpson/permute@b0bed3cbb6517a1ee13f6fb357ba9e6db385f69b 🚀
  document changes for #41; bump patch version again
  Deploying to gh-pages from @ gavinsimpson/permute@688cca6db016e71faf1043909732fd78b87e3657 🚀

=== openclimatefix_satip ===
  Update README.md
  Bump version: 2.12.39 → 2.12.40 [skip ci]
  update example cord transform function
  [pre-commit.ci] pre-commit autoupdate

updates:
- [github.com/pre-commit/pre-commit-hooks: v4.5.0 → 
  [pre-commit.ci] auto fixes from pre-commit.com hooks

for more information, see https://pre-commit.c
  Update deprecation notice i

In [9]:
for proj in projects:
    dfp = df_gap_commits[df_gap_commits['project_wocid'] == proj]
    pre_authors = set(dfp[dfp['pre_post'] == 'pre']['author'])
    post_authors = set(dfp[dfp['pre_post'] == 'post']['author'])
    overlap = pre_authors & post_authors

    print(f"=== {proj} ===")
    print(" pre authors: ", pre_authors)
    print(" post authors:", post_authors)
    print(" same people?:", "YES" if overlap else "NO — new contributors")
    print()

=== gavinsimpson_permute ===
 pre authors:  {'Gavin Simpson <ucfagls@gmail.com>'}
 post authors: {'Gavin Simpson <ucfagls@gmail.com>'}
 same people?: YES

=== openclimatefix_satip ===
 pre authors:  {'Rabscuttler <laurence.watson@gmail.com>', 'AyrtonB <AyrtonBourn@Outlook.com>', 'Jacob Bieker <jacob@openclimatefix.org>'}
 post authors: {'Jacob Bieker <jacob@bieker.tech>'}
 same people?: NO — new contributors

=== juliaopt_convex.jl ===
 pre authors:  {'Theo Diamandis <tdiamand@mit.edu>', 'Theo Diamandis <tjdiamandis@gmail.com>', 'Oscar Dowson <odow@users.noreply.github.com>', 'Alan Edelman <mit.edelman@gmail.com>', 'odow <o.dowson@gmail.com>'}
 post authors: {'Eric Hanson <5846501+ericphanson@users.noreply.github.com>'}
 same people?: NO — new contributors

=== eljost_pysisyphus ===
 pre authors:  {'Johannes Steinmetzer <johannes.steinmetzer@uni-jena.de>', 'Johannes Steinmetzer <jsteinmetzer1108@aol.com>'}
 post authors: {'Johannes Steinmetzer <jsteinmetzer1108@aol.com>'}
 same people?

In [10]:
part4_data = {
    'gavinsimpson_permute': {
        'HypothesizedGapReason': "Solo-maintained R package; pre-gap commits were entirely reactive fixes for an R 3.6.0 RNG internals change, suggesting the maintainer paused once that issue was resolved and no new work was pending.",
        'HasRecovered': 'Partially',
        'WhyRecovered': "Same maintainer returned to address a specific bug report (#28) and do periodic CRAN-compliance maintenance (GH Actions, vignette format).",
        'WhoRecovered': 'Same person (Gavin Simpson) throughout',
        'RecentThemes': 'Documentation updates:Release',
        'Notes': "Declining, solo-maintained R package. The 31-month gap was easy to interpret -- commits before it were reactive fixes for an R internals change, and commits after addressed a specific bug report and CRAN maintenance tasks. Same single author throughout; no new contributors.",
    },
    'openclimatefix_satip': {
        'HypothesizedGapReason': 'N/A active Project',
        'HasRecovered': 'N/A',
        'WhyRecovered': 'N/A',
        'WhoRecovered': 'N/A',
        'RecentThemes': 'Automated bot contributions:Release',
        'Notes': "N/A -- no inactivity gap >=3 months exists; Satip stayed continuously active. Bot-driven commits (pre-commit.ci) likely smooth over what would otherwise look like dead months. Contributor base narrowed over time but didn't disappear.",
    },
    'juliaopt_convex.jl': {
        'HypothesizedGapReason': "Pre-gap commits show normal routine maintenance with no sign of distress; the gap itself has no obvious trigger visible in the data.",
        'HasRecovered': 'Yes, briefly',
        'WhyRecovered': "A new contributor (Eric Hanson) took over and drove an intense refactor/simplification effort right after the gap.",
        'WhoRecovered': 'New contributor -- none of the pre-gap authors appear post-gap',
        'RecentThemes': 'Documentation updates:Other',
        'Notes': "Irregular pattern; 4-month gap followed by a short, intense refactor burst led entirely by a new contributor (Eric Hanson), not the original authors. Appears to be a maintainer handoff rather than a genuine organic revival -- activity dropped again shortly after.",
    },
    'eljost_pysisyphus': {
        'HypothesizedGapReason': 'N/A active Project',
        'HasRecovered': 'N/A',
        'WhyRecovered': 'N/A',
        'WhoRecovered': 'N/A',
        'RecentThemes': 'Other:Bug fixes',
        'Notes': "N/A -- no inactivity gap >=3 months; continuously high-volume activity from a single primary author (Johannes Steinmetzer) for most of its history. A recent (late 2025) slowdown is visible but too short to count as a formal gap yet.",
    },
    'simonvh_genomepy': {
        'HypothesizedGapReason': "Pre-gap commits are routine release/deployment maintenance with no sign of distress.",
        'HasRecovered': 'Partially',
        'WhyRecovered': "A broken MySQL dependency forced a reactive fix -- a classic external-trigger recovery rather than organic new development.",
        'WhoRecovered': 'Mostly same person (siebrenf), joined by one new contributor (Maarten-vd-Sande)',
        'RecentThemes': 'Release:Other',
        'Notes': "Declining. The 9-month gap ended with a reactive dependency fix (broken MySQL connection), not new feature work. Same core maintainer returned, joined by one additional contributor. Now in low-volume maintenance mode.",
    },
    'brentp_cyvcf2': {
        'HypothesizedGapReason': "No clear distress signal pre-gap; this project has always been sparse (3 separate 3+ month gaps historically).",
        'HasRecovered': 'Yes',
        'WhyRecovered': "A new Python release (3.13/3.14) required CI compatibility updates -- a reactive, ecosystem-driven trigger.",
        'WhoRecovered': 'Same core maintainer (Brent Pedersen), joined by one new contributor (Ben Jeffery)',
        'RecentThemes': 'Dependency updates:Release',
        'Notes': "Irregular, historically sparse. Most recent 11-month gap ended when a new Python release (3.13/3.14) forced compatibility updates. Core maintainer (Brent Pedersen) still present, joined by one new contributor for the CI work.",
    },
    'dicarlolab-delft_pycqed_py3': {
        'HypothesizedGapReason': 'N/A active Project',
        'HasRecovered': 'N/A',
        'WhyRecovered': 'N/A',
        'WhoRecovered': 'N/A',
        'RecentThemes': 'Other:Dependency updates',
        'Notes': "N/A -- no inactivity gap >=3 months despite an overall declining trend. Commit messages are largely uninformative (many blank/unlabeled merges), making fine-grained interpretation difficult. Same author present throughout the analyzed window.",
    },
    'benchmarking-initiative_benchmark-models-petab': {
        'HypothesizedGapReason': "The one recorded gap sits at the very start of the project's real activity, before 81% of all commits happened -- reads as an early startup lull, not abandonment.",
        'HasRecovered': 'Yes',
        'WhyRecovered': "The project found its steady community-contribution rhythm shortly after, with people periodically joining to add their own models.",
        'WhoRecovered': 'Mix -- some continuity (dilpath) plus several new contributors (vanako, PaulJonasJost, IvaEwert)',
        'RecentThemes': 'Feature development:Other',
        'Notes': "Cyclical, consistently community-driven. The one 3-month gap happened early, before the project's real growth, and reads as a startup lull. New contributors join regularly to add their own models, alongside some persistent core contributors.",
    },
    'gsdlab_clafer': {
        'HypothesizedGapReason': "Original small team (Klaas Gadeyne, Michal Antkiewicz, toeklk) appears to have dispersed after 2015's activity tapered off.",
        'HasRecovered': 'Barely',
        'WhyRecovered': "Needed to keep the project buildable on a newer Haskell/GHC toolchain -- purely reactive maintenance, not new development.",
        'WhoRecovered': 'Same lead researcher (Michal Antkiewicz) returned alone, under a new institutional email, without the rest of the original team',
        'RecentThemes': 'Other:Dependency updates',
        'Notes': "Sharpest decline studied -- 50-month gap, the longest of all 10 projects, with only ~1% of commits occurring afterward. Same lead researcher (Michal Antkiewicz) returned alone, without the rest of the original team, purely to do toolchain-compatibility maintenance.",
    },
    'acellera_htmd': {
        'HypothesizedGapReason': "No dramatic change in work type around the gap -- both sides show routine bug-fix/parameterization maintenance, suggesting a normal short lull rather than a crisis.",
        'HasRecovered': 'Yes',
        'WhyRecovered': "Continued routine maintenance work (atom typing, deprecation fixes) resumed with no major disruption visible.",
        'WhoRecovered': 'Same core maintainer (Stefan Doerr), joined by one new contributor (Adria Perez)',
        'RecentThemes': 'Bug fixes:Dependency updates',
        'Notes': "Declining from an intense 2016-2019 peak into steady low-volume maintenance. The one 3-month gap was a normal lull -- both sides show routine maintenance work, no major disruption. Core maintainer present throughout, joined by one new contributor.",
    },
}

df_stats['HypothesizedGapReason'] = df_stats['Project'].map(lambda p: part4_data[p]['HypothesizedGapReason'])
df_stats['HasRecovered'] = df_stats['Project'].map(lambda p: part4_data[p]['HasRecovered'])
df_stats['WhyRecovered'] = df_stats['Project'].map(lambda p: part4_data[p]['WhyRecovered'])
df_stats['WhoRecovered'] = df_stats['Project'].map(lambda p: part4_data[p]['WhoRecovered'])
df_stats['RecentThemes'] = df_stats['Project'].map(lambda p: part4_data[p]['RecentThemes'])
df_stats['Notes'] = df_stats['Project'].map(lambda p: part4_data[p]['Notes'])

# Currentstatus: rule is inactive only if no commits since April 1 2025.
# All 10 projects have lastGHCommitDate after that, so all are Active.
df_stats['Currentstatus'] = 'Active'

df_stats.to_csv('tlopez7_project_stats.csv', index=False)
print(df_stats[['Project', 'Currentstatus', 'HasRecovered', 'WhoRecovered']])

                                          Project Currentstatus  HasRecovered  \
0                            gavinsimpson_permute        Active     Partially   
1                            openclimatefix_satip        Active           N/A   
2                              juliaopt_convex.jl        Active  Yes, briefly   
3                               eljost_pysisyphus        Active           N/A   
4                                simonvh_genomepy        Active     Partially   
5                                   brentp_cyvcf2        Active           Yes   
6                     dicarlolab-delft_pycqed_py3        Active           N/A   
7  benchmarking-initiative_benchmark-models-petab        Active           Yes   
8                                   gsdlab_clafer        Active        Barely   
9                                   acellera_htmd        Active           Yes   

                                        WhoRecovered  
0             Same person (Gavin Simpson) throughout 

In [11]:
reflections = {
    'gavinsimpson_permute': """gavinsimpson/permute shows a declining trajectory, with dense activity from 2010-2016 (14 separate gaps of 3+ months even during its active years) tapering into near-total dormancy afterward. The longest gap (2019-04 to 2021-10, 31 months) was relatively easy to interpret: the commits immediately before the gap were entirely about updating tests for a change to R's random number generator in R 3.6.0, suggesting the maintainer paused once that compatibility issue was resolved. The commits immediately after address a specific bug report (#28) and modernize the package's CI/documentation tooling -- signs of periodic CRAN-compliance maintenance rather than active feature development. The single, consistent author throughout (Gavin Simpson) confirms this is a one-person academic package kept alive in occasional bursts.""",

    'openclimatefix_satip': """openclimatefix/Satip shows an irregular activity pattern with no formal inactivity gap of 3+ months anywhere in its 5-year history -- the technically 'longest' dip was only 2 months, under the threshold for a real gap. This makes it an actively, if unevenly, maintained project rather than one with a recovery story. The mix of automated bot commits (pre-commit.ci autoupdates, version bumps) alongside genuine development work suggests a small, consistently engaged team using CI automation to keep the project ticking over even in quieter months. Authorship narrowed over time (from a 3-person group to primarily Jacob Bieker more recently), which may explain volatility in commit volume without representing an actual abandonment-and-recovery cycle.""",

    'juliaopt_convex.jl': """JuliaOpt/Convex.jl shows an irregular pattern: years of modest, routine activity were interrupted by a 4-month gap (2023-03 to 2023-06), after which the project saw an unusual burst of very large commits focused on simplification and removing old code variants. This gap was relatively easy to interpret -- the pre-gap commits look like normal maintenance with no sign of distress, while post-gap commits ('simplify', 'remove more complicated variants') clearly describe an internal refactor, not new feature growth. Notably, none of the pre-gap authors appear in the post-gap commits -- a new contributor (Eric Hanson) took over, suggesting a maintainer handoff rather than the original team returning. Recent commits suggest the burst was a one-time cleanup rather than a sustained revival.""",

    'eljost_pysisyphus': """eljost/pysisyphus shows no real inactivity gap -- the longest stretch of zero commits was just 1 month, and the project sustained high, volatile activity (50-300 commits/month) for nearly its entire 8-year history. This is a case where gap analysis isn't really applicable; the project is best described by its overall irregular pattern. The one consistent thread is a single primary author (Johannes Steinmetzer) across the commits examined, suggesting a single-researcher scientific computing project developed in bursts tied to research needs rather than a team-maintained package with distinct dormant periods. A genuine slowdown appears in late 2025, worth monitoring even though it doesn't meet the 3-month gap threshold yet.""",

    'simonvh_genomepy': """simonvh/genomepy shows a declining trajectory: strong, spiky activity from 2019-2022 gives way to a 9-month gap (2023-12 to 2024-08) and only a trickle afterward (3% of all commits). The gap was moderately easy to interpret -- commits right before it are release/deployment maintenance with no sign of distress, while commits right after fix a broken MySQL dependency, a classic external-trigger recovery rather than organic development. The same core maintainer (siebrenf) appears both before and after, joined by one additional contributor (Maarten-vd-Sande) post-gap, indicating a small, dedicated team doing reactive maintenance on a mostly-finished tool.""",

    'brentp_cyvcf2': """brentp/cyvcf2 has been sparse and inconsistent for its entire lifespan, with 3 separate gaps of 3+ months across its history. The most recent and longest (2024-10 to 2025-08, 11 months) was easy to interpret: commits immediately after are entirely about adding CI support for new Python versions (3.13, 3.14) -- the maintainer returned because a new Python release required compatibility updates, not organic new development. The primary maintainer (Brent Pedersen) is present in both pre- and post-gap commits, joined by a new contributor (Ben Jeffery) after the gap, suggesting the core maintainer relies on periodic help to keep pace with the Python ecosystem's release cycle.""",

    'dicarlolab-delft_pycqed_py3': """DiCarloLab-Delft/PycQED_py3 shows a clear declining trend overall (a huge 2017 peak tapering to a low trickle by 2025), but technically has zero formal inactivity gaps of 3+ months -- even in its quietest periods, at least one commit sneaks in every month or two. This makes the gap analysis somewhat misleading: the algorithm found a 2-month dip as the 'longest gap,' but the real story is a long, slow decline rather than a distinct dormant-then-revived cycle. Commits around this technical gap are dominated by merge commits with little descriptive content -- this project's own commit hygiene limits fine-grained interpretation. The same primary author (Marios Samiotis) appears throughout, consistent with an academic lab codebase maintained by whichever researcher is actively using it.""",

    'benchmarking-initiative_benchmark-models-petab': """Benchmarking-Initiative/Benchmark-Models-PEtab shows a cyclical pattern -- recurring bursts of model-contribution activity every few months with no clear long-term trend, just as active in late 2025 as in 2021. Its one recorded 3-month gap (2020-07 to 2020-09) sits right at the start of its real activity (before 81% of all commits happened), reading as an early lull before the project found its steady community rhythm rather than a sign of abandonment. The contributor base shows healthy turnover -- some names persist across the gap while several new contributors appear afterward -- consistent with a community benchmark repository where people periodically join to contribute their own model.""",

    'gsdlab_clafer': """gsdlab/clafer shows the most dramatic decline of all 10 projects studied: heavy team activity from 2011-2015 is followed by a 50-month gap (2019-06 to 2023-07) -- the longest gap in the entire dataset -- after which only about 1% of all commits occurred. This gap was easy to interpret from the author data: pre-gap commits came from a small team, while post-gap commits come from only one person (Michal Antkiewicz, under a new institutional email), meaning the project's lead researcher returned alone after the rest of the original contributors moved on. The post-gap commits (upgrading to a newer Haskell/GHC toolchain) are purely reactive maintenance to keep the project buildable, not new development -- this looks like an academic research project kept minimally alive by its original lead long after active development ended.""",

    'acellera_htmd': """Acellera/htmd shows a clear boom-and-fade pattern: intense activity from 2016-2019 settles into a persistent low trickle for 2020-2025, including a 3-month gap (2023-07 to 2023-09). This gap was easy to interpret: both pre- and post-gap commits are routine bug-fix and parameterization maintenance, with no dramatic change in the type of work being done -- a normal short lull in a mature, still-maintained tool rather than a true abandonment scare. The primary maintainer (Stefan Doerr) is present throughout, joined by a new contributor (Adria Perez) after the gap, suggesting a small, stable core team still actively supports the software even though its most active development period is well behind it.""",
}

for proj, text in reflections.items():
    with open(f'tlopez7_reflection_{proj}.md', 'w') as f:
        f.write(f"# Reflection: {proj}\n\n{text.strip()}\n")

print("Wrote", len(reflections), "reflection files")

Wrote 10 reflection files
